## Data extraction for a reproducible VQAv2 validation subset

Produces a 2,000-question VQAv2 validation subset and extracts only its referenced COCO val2014 images from `val2014.zip`.

The following files are supposed to be within `data/`:
- `v2_OpenEnded_mscoco_val2014_questions.json`
- `v2_mscoco_val2014_annotations.json`
- `val2014.zip`
  
The outputs are under `data/validation_dataset/`. 

In [1]:
from pathlib import Path
import csv
import json
import random
import shutil
import zipfile
from pathlib import PurePosixPath

working_dir = Path.cwd().resolve()
PROJECT_ROOT = working_dir.parent if working_dir.name.lower() == "scripts" else working_dir
DATA_DIR = PROJECT_ROOT / "data"

QUESTIONS_JSON = DATA_DIR / "v2_OpenEnded_mscoco_val2014_questions.json"
ANNOTATIONS_JSON = DATA_DIR / "v2_mscoco_val2014_annotations.json"
IMAGES_ZIP = DATA_DIR / "val2014.zip"
OUTPUT_DIR = DATA_DIR / "validation_dataset"

NUM_QUESTIONS = 2000
RANDOM_SEED = 42
OVERWRITE_IMAGES = False

In [2]:
def prepare_vqav2_subset(
    questions_path: Path,
    annotations_path: Path,
    images_zip_path: Path,
    output_dir: Path,
    num_questions: int = 2000,
    seed: int = 42,
    overwrite_images: bool = False,
) -> dict:
    """Select annotated VQAv2 validation questions and extract only their images."""
    if num_questions < 1:
        raise ValueError("num_questions must be positive.")
    for path in (questions_path, annotations_path, images_zip_path):
        if not path.is_file():
            raise FileNotFoundError(path)

    with questions_path.open("r", encoding="utf-8") as file:
        questions_data = json.load(file)
    with annotations_path.open("r", encoding="utf-8") as file:
        annotations_data = json.load(file)

    questions = questions_data["questions"]
    annotations = annotations_data["annotations"]
    if not isinstance(questions, list) or not isinstance(annotations, list):
        raise ValueError("Expected VQAv2 questions and annotations lists.")

    annotations_by_id = {int(a["question_id"]): a for a in annotations}
    if len(annotations_by_id) != len(annotations):
        raise ValueError("Duplicate question IDs found in annotations.")

    eligible = [q for q in questions if int(q["question_id"]) in annotations_by_id]
    if num_questions > len(eligible):
        raise ValueError(f"Requested {num_questions} questions, but only {len(eligible)} are annotated.")

    # Sample before sorting to match the original notebook's seed-42 selection.
    selected_questions = random.Random(seed).sample(eligible, num_questions)
    selected_questions.sort(key=lambda q: int(q["question_id"]))
    selected_ids = [int(q["question_id"]) for q in selected_questions]
    if len(set(selected_ids)) != num_questions:
        raise ValueError("Duplicate question IDs found in the selected questions.")
    selected_annotations = [annotations_by_id[qid] for qid in selected_ids]

    def image_name(image_id: int) -> str:
        return f"COCO_val2014_{image_id:012d}.jpg"

    wanted_images = {image_name(int(q["image_id"])) for q in selected_questions}
    image_dir = output_dir / "images" / "val2014"
    image_dir.mkdir(parents=True, exist_ok=True)
    extracted = 0
    reused = 0

    with zipfile.ZipFile(images_zip_path) as archive:
        # Read ZIP metadata only; image contents are streamed one at a time.
        entries = {}
        for info in archive.infolist():
            if info.is_dir():
                continue
            name = PurePosixPath(info.filename.replace("\\", "/")).name
            if name in wanted_images:
                if name in entries:
                    raise ValueError(f"Duplicate COCO image entry in ZIP: {name}")
                entries[name] = info

        missing = sorted(wanted_images - entries.keys())
        if missing:
            raise FileNotFoundError(f"ZIP is missing {len(missing)} required images; first: {missing[0]}")

        for filename in sorted(wanted_images):
            info = entries[filename]
            destination = image_dir / filename
            # Compare size too: a prior interrupted extraction must not count as complete.
            if (not overwrite_images and destination.is_file()
                    and destination.stat().st_size == info.file_size):
                reused += 1
                continue

            partial = image_dir / f".{filename}.partial"
            try:
                with archive.open(info) as source, partial.open("wb") as target:
                    shutil.copyfileobj(source, target, length=1024 * 1024)
                if partial.stat().st_size != info.file_size:
                    raise IOError(f"Incomplete extraction: {filename}")
                partial.replace(destination)
            finally:
                partial.unlink(missing_ok=True)
            extracted += 1

    questions_out = output_dir / "v2_OpenEnded_mscoco_val2014_questions_subset.json"
    annotations_out = output_dir / "v2_mscoco_val2014_annotations_subset.json"
    manifest_out = output_dir / "subset_manifest.csv"

    with questions_out.open("w", encoding="utf-8") as file:
        json.dump({**questions_data, "questions": selected_questions}, file, ensure_ascii=False, indent=2)
    with annotations_out.open("w", encoding="utf-8") as file:
        json.dump({**annotations_data, "annotations": selected_annotations}, file, ensure_ascii=False, indent=2)

    with manifest_out.open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(("question_id", "image_id", "image_filename"))
        for question in selected_questions:
            image_id = int(question["image_id"])
            writer.writerow((int(question["question_id"]), image_id, image_name(image_id)))

    return {
        "questions": len(selected_questions),
        "unique_images": len(wanted_images),
        "extracted": extracted,
        "reused": reused,
        "output_dir": output_dir,
    }

In [3]:
result = prepare_vqav2_subset(
    questions_path=QUESTIONS_JSON,
    annotations_path=ANNOTATIONS_JSON,
    images_zip_path=IMAGES_ZIP,
    output_dir=OUTPUT_DIR,
    num_questions=NUM_QUESTIONS,
    seed=RANDOM_SEED,
    overwrite_images=OVERWRITE_IMAGES,
)

print(
    f"{result['questions']:,} questions, {result['unique_images']:,} unique images "
    f"({result['extracted']:,} extracted, {result['reused']:,} reused)."
)

2,000 questions, 1,931 unique images (1,931 extracted, 0 reused).
